# Compare PhaseNet Models: Zhu et al. (2019) vs v7

This notebook compares the original PhaseNet (Zhu & Beroza, 2019) against v7 (fine-tuned on hybrid global dataset with knowledge distillation) on the same Ridgecrest waveforms.

**Goal**: Quantify differences in detection rate, timing accuracy, and false positives.

In [ ]:
import obspy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seisbench.models as sbm
import seisbench.util as sbu
from s3fs import S3FileSystem
from tqdm.notebook import tqdm
from datetime import datetime, timedelta

: 

## Configuration

In [ ]:
# Ridgecrest M7.1 earthquake
EVENT_TIME = obspy.UTCDateTime("2019-07-05T17:33:50Z")
EVENT_LAT = 35.705
EVENT_LON = -117.504

# SCSN stations within 50 km of epicenter
# All from Southern California Seismic Network (CI network)
# Data source: SCEDC S3 bucket (scedc-pds/continuous_waveforms/)
TEST_STATIONS = [
    ("CI", "DAM"),    # Furnace Creek, 6 km NW
    ("CI", "BOR"),    # Boron, 17 km N
    ("CI", "SYC"),    # Sycamore, 34 km NNW
    ("CI", "TNP"),    # Timberlake Peak, 39 km N
    ("CI", "PIG"),    # Pisgah area, 48 km NW
]

# Event day
year = 2019
doy = 186

## Step 1: Fetch waveforms

In [ ]:
fs = S3FileSystem(anon=True)

waveforms = {}

for net, sta in TEST_STATIONS:
    print(f"\nFetching {net}.{sta}...")
    stream = obspy.Stream()
    
    for channel in "ZNE":
        # SCEDC S3 bucket (Southern California Earthquake Data Center)
        s3_path = f"scedc-pds/continuous_waveforms/{net}/{year}/{year}.{doy:03d}/{sta}.{net}.HH{channel}.00.D.{year}.{doy:03d}"
        
        try:
            with fs.open(s3_path) as f:
                trace = obspy.read(f)
                stream += trace
        except Exception as e:
            print(f"  ✗ {channel}: {type(e).__name__}")
    
    if len(stream) > 0:
        t0 = min(t.stats.starttime for t in stream)
        stream.trim(starttime=t0, endtime=t0 + 24 * 3600)
        waveforms[f"{net}.{sta}"] = stream
        print(f"  → Stored {len(stream)} traces")

print(f"\nLoaded {len(waveforms)} stations")

## Step 2: Load both models

**Models being compared:**

1. **original** (Zhu & Beroza, 2019)  
   - Trained on SCSN data only  
   - Published baseline

2. **v7** (Denolle Lab, 2026)  
   - Fine-tuned from jma_wc (Japanese regional)  
   - Trained on hybrid ~20-dataset corpus  
   - Knowledge distillation from jma_wc teacher (alpha=0.3, T=4.0)  
   - Focuses on detection recall (timing_beta=0)  
   - Benchmark (cross-domain): P-MAE=0.340s, P-recall=0.853, MCC=0.760

In [ ]:
models = {}
model_info = {}

# Load original Zhu et al. 2019
try:
    print("Loading Zhu et al. (2019) original PhaseNet...")
    models['original'] = sbm.PhaseNet.from_pretrained("original")
    model_info['original'] = {
        'name': 'PhaseNet (Zhu & Beroza, 2019)',
        'description': 'Original baseline trained on SCSN',
        'dataset': 'SCSN',
        'distillation': 'None'
    }
    print("  ✓ Loaded")
except Exception as e:
    print(f"  ✗ Error: {e}")

# Load v7 (quakescope2026)
try:
    print("\nLoading v7 (quakescope2026)...")
    models['v7'] = sbm.PhaseNet.from_pretrained("quakescope2026")
    model_info['v7'] = {
        'name': 'PhaseNet v7 (Denolle Lab, 2026)',
        'description': 'Fine-tuned from jma_wc, knowledge distillation',
        'dataset': 'Hybrid (~20 SeisBench datasets)',
        'distillation': 'Yes (alpha=0.3, T=4.0)'
    }
    print("  ✓ Loaded")
except Exception as e:
    print(f"  ⚠ v7 not available: {e}")
    print("  Falling back to 'jma_wc' (Japanese) for comparison...")
    models['v7'] = sbm.PhaseNet.from_pretrained("jma_wc")
    model_info['v7'] = {
        'name': 'PhaseNet jma_wc (SeisBench, 2021)',
        'description': 'Japanese regional network model (v7 parent)',
        'dataset': 'Japanese seismic network',
        'distillation': 'No'
    }
    print("  ✓ Loaded jma_wc as fallback")

print(f"\n=== Model Comparison ===")
for model_name, info in model_info.items():
    print(f"\n{model_name.upper()}:")
    for key, val in info.items():
        print(f"  {key}: {val}")

## Step 3: Run inference on both models

In [ ]:
results = {model_name: {} for model_name in models.keys()}

P_THRESHOLD = 0.3
S_THRESHOLD = 0.3

for model_name, model in models.items():
    print(f"\n=== {model_info[model_name]['name']} ===")
    
    all_picks = sbu.PickList()
    station_picks = {}
    
    for sta_id, stream in tqdm(waveforms.items(), desc=f"Running {model_name}"):
        try:
            pred = model.classify(stream, P_threshold=P_THRESHOLD, S_threshold=S_THRESHOLD)
            picks = pred.picks
            all_picks += picks
            station_picks[sta_id] = picks
        except Exception as e:
            print(f"  ✗ {sta_id}: {e}")
    
    results[model_name]['all_picks'] = all_picks
    results[model_name]['station_picks'] = station_picks
    
    # Count picks by phase
    df = all_picks.df
    p_count = len(df[df['phase'] == 'P'])
    s_count = len(df[df['phase'] == 'S'])
    
    print(f"\nTotal picks: {len(all_picks)} (P: {p_count}, S: {s_count})")
    print(f"\nPicks per station:")
    for sta_id, picks in station_picks.items():
        p = len([pick for pick in picks if 'P' in str(pick.phase)])
        s = len([pick for pick in picks if 'S' in str(pick.phase)])
        print(f"  {sta_id}: P={p}, S={s}")

## Step 4: Comparison metrics

In [ ]:
# Build comparison table
comparison_data = []

for model_name in models.keys():
    picks = results[model_name]['all_picks']
    df = picks.df
    
    p_picks = df[df['phase'] == 'P']
    s_picks = df[df['phase'] == 'S']
    
    comparison_data.append({
        'Model': model_info[model_name]['name'],
        'Total Picks': len(picks),
        'P Picks': len(p_picks),
        'S Picks': len(s_picks),
        'Mean P Confidence': p_picks['peak_value'].mean() if len(p_picks) > 0 else np.nan,
        'Mean S Confidence': s_picks['peak_value'].mean() if len(s_picks) > 0 else np.nan,
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n=== COMPARISON TABLE ===")
print(comparison_df.to_string(index=False))

# Calculate percent differences
if len(models) == 2:
    model_names = list(models.keys())
    original_picks = results[model_names[0]]['all_picks']
    v7_picks = results[model_names[1]]['all_picks']
    
    pct_diff = (len(v7_picks) - len(original_picks)) / len(original_picks) * 100 if len(original_picks) > 0 else 0
    print(f"\nv7 vs {model_names[0]}: {pct_diff:+.1f}% difference in total picks")

## Step 5: Visualize picks for each model

In [ ]:
def plot_model_picks(stream, picks, model_name, event_time, before=10, after=60):
    """
    Plot 3-component waveform with picks for a single model.
    """
    fig, axes = plt.subplots(3, 1, figsize=(14, 8))
    
    channels = ["HHZ", "HHN", "HHE"]
    
    for ax, channel in zip(axes, channels):
        tr = stream.select(channel=channel)
        if not tr:
            continue
        
        tr = tr[0]
        t_start = tr.stats.starttime
        dt = tr.stats.delta
        
        time_vec = np.arange(len(tr)) * dt
        ax.plot(time_vec, tr.data, 'k-', linewidth=0.5, alpha=0.7)
        
        # Event time
        event_offset = (event_time - t_start)
        ax.axvline(event_offset, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Event')
        
        # Picks
        for pick in picks:
            pick_offset = (pick.time - t_start)
            color = 'blue' if 'P' in str(pick.phase) else 'green'
            ax.axvline(pick_offset, color=color, linewidth=2, alpha=0.8)
            idx = int(pick_offset / dt)
            if 0 <= idx < len(tr):
                ax.plot(pick_offset, tr.data[idx], marker='o', color=color, markersize=8)
        
        # Event window
        ax.axvspan(event_offset - before, event_offset + after, alpha=0.1, color='gray')
        
        ax.set_ylabel(channel)
        ax.grid(True, alpha=0.3)
    
    axes[-1].set_xlabel("Time since 00:00 UTC (s)")
    fig.suptitle(f"{model_name}", fontsize=14, fontweight='bold')
    
    return fig

# Plot for each station and model
for sta_id in waveforms.keys():
    stream = waveforms[sta_id]
    
    # Side-by-side comparison
    fig, axes = plt.subplots(3, len(models), figsize=(16, 8))
    if len(models) == 1:
        axes = axes.reshape(-1, 1)
    
    channels = ["HHZ", "HHN", "HHE"]
    
    for col, (model_name, model) in enumerate(models.items()):
        picks = results[model_name]['station_picks'].get(sta_id, [])
        
        for row, channel in enumerate(channels):
            ax = axes[row, col]
            tr = stream.select(channel=channel)
            
            if not tr:
                continue
            
            tr = tr[0]
            t_start = tr.stats.starttime
            dt = tr.stats.delta
            
            time_vec = np.arange(len(tr)) * dt
            ax.plot(time_vec, tr.data, 'k-', linewidth=0.5, alpha=0.7)
            
            # Event time
            event_offset = (EVENT_TIME - t_start)
            ax.axvline(event_offset, color='red', linestyle='--', linewidth=2, alpha=0.5)
            
            # Picks
            for pick in picks:
                pick_offset = (pick.time - t_start)
                color = 'blue' if 'P' in str(pick.phase) else 'green'
                ax.axvline(pick_offset, color=color, linewidth=2, alpha=0.8)
            
            ax.set_ylabel(channel if col == 0 else '')
            ax.grid(True, alpha=0.3)
            
            if row == 0:
                ax.set_title(f"{model_info[model_name]['name']}", fontsize=12, fontweight='bold')
    
    fig.suptitle(f"Model Comparison: {sta_id}", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Step 6: Key Findings

### Expected Differences

**v7 vs. Original:**

- **Detection rate**: v7 trained on global data (20+ datasets), should generalize better to varied network types
- **P-recall**: Original optimized for SCSN. v7's benchmark shows 0.853 P-recall on cross-domain split
- **Timing**: Original has precise timing on regional events. v7 emphasizes detection (timing_beta=0, no timing loss)
- **False positives**: v7 may have higher false-positive rate in distant/noisy regions due to global training
- **Domain gap**: v7 is NOT fine-tuned on SCSN; it's trained on jma_wc (Japanese) as parent → may behave differently on California stations

In [ ]:
print("\n=== INTERPRETATION GUIDE ===")
print("""
To validate v7 is production-ready vs. original on close-field data:

1. CHECK: Do pick counts make sense?
   - v7 may detect more events (higher recall) or fewer (more selective)
   - Both should cluster picks near 17:33:50 UTC

2. CHECK: Do picks align with waveform onsets?
   - Both models should mark real P and S arrivals
   - Spurious picks indicate poor generalization

3. CHECK: Are P-S intervals physical?
   
   All 5 stations are close (<50 km) to epicenter.
   Expected P-S delays (Vp ≈ 5.8 km/s, Vs ≈ 3.3 km/s):
   
   Station  | Distance | Expected P-S (s)
   ---------|----------|------------------
   DAM      | 6 km     | 0.7 - 1.5s
   BOR      | 17 km    | 2.0 - 3.2s
   SYC      | 34 km    | 4.0 - 6.0s
   TNP      | 39 km    | 4.6 - 7.0s
   PIG      | 48 km    | 5.7 - 8.6s
   
   All values should be short (close range) with tight clustering.

4. DECISION:
   - If v7 detects same events with aligned onsets on all stations → PASS (ready for production)
   - If v7 misses obvious arrivals → INVESTIGATE (timing loss or domain gap?)
   - If v7 spurious picks in noise → CAUTION (may need local fine-tuning)
   - If results consistent across all 5 close stations → HIGH CONFIDENCE in deployment
""")